# Assignment 10: N-gram Auto-complete Algorithm

## Objective
Develop an improved auto-complete algorithm using N-gram models similar to those used in translation, authorship detection, and speech recognition.

## 1. Install and Import Libraries

In [ ]:
%pip install nltk numpy pandas
import nltk
from nltk import ngrams, FreqDist
from nltk.tokenize import word_tokenize
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('brown', quiet=True)

print("Libraries imported successfully!")

## 2. Training Corpus

Creating a training corpus for building the N-gram model.

In [ ]:
# Training corpus - sample text data
training_corpus = """
Natural language processing is a subfield of artificial intelligence.
Machine learning is used in natural language processing.
Deep learning models have revolutionized natural language processing.
The quick brown fox jumps over the lazy dog.
Artificial intelligence and machine learning are transforming the world.
Natural language understanding is a key challenge in artificial intelligence.
Speech recognition systems use natural language processing techniques.
Machine translation is an important application of natural language processing.
Text mining and information extraction rely on natural language processing.
Sentiment analysis is a popular natural language processing task.
Language models predict the next word in a sequence.
Neural networks are widely used in natural language processing.
Word embeddings capture semantic meaning in natural language processing.
Transformers have become the standard architecture for natural language processing.
BERT and GPT are powerful language models.
"""

# Tokenize and preprocess
tokens = word_tokenize(training_corpus.lower())
print("Training Corpus Statistics:")
print("=" * 70)
print(f"Total tokens: {len(tokens)}")
print(f"Unique tokens: {len(set(tokens))}")
print(f"\nFirst 30 tokens: {tokens[:30]}")

## 3. Build N-gram Models

Creating unigram, bigram, and trigram models.

In [ ]:
# Generate N-grams
unigrams = list(ngrams(tokens, 1))
bigrams = list(ngrams(tokens, 2))
trigrams = list(ngrams(tokens, 3))

# Count frequencies
unigram_freq = Counter(unigrams)
bigram_freq = Counter(bigrams)
trigram_freq = Counter(trigrams)

print("N-gram Statistics:")
print("=" * 70)
print(f"Total unigrams: {len(unigram_freq)}")
print(f"Total bigrams: {len(bigram_freq)}")
print(f"Total trigrams: {len(trigram_freq)}")

print("\nTop 10 Unigrams:")
for gram, count in unigram_freq.most_common(10):
    print(f"  {gram[0]:20s} : {count}")

print("\nTop 10 Bigrams:")
for gram, count in bigram_freq.most_common(10):
    print(f"  {' '.join(gram):35s} : {count}")

print("\nTop 10 Trigrams:")
for gram, count in trigram_freq.most_common(10):
    print(f"  {' '.join(gram):50s} : {count}")

## 4. Bigram Language Model

Building a bigram model for word prediction.

In [ ]:
class BigramModel:
    def __init__(self):
        self.bigram_counts = defaultdict(Counter)
        self.unigram_counts = Counter()
    
    def train(self, tokens):
        """Train the bigram model"""
        # Count unigrams
        for token in tokens:
            self.unigram_counts[token] += 1
        
        # Count bigrams
        for i in range(len(tokens) - 1):
            self.bigram_counts[tokens[i]][tokens[i+1]] += 1
    
    def predict_next(self, word, top_k=5):
        """Predict next words given current word"""
        if word not in self.bigram_counts:
            return []
        
        # Get possible next words with their probabilities
        next_words = self.bigram_counts[word]
        total = sum(next_words.values())
        
        predictions = []
        for next_word, count in next_words.most_common(top_k):
            probability = count / total
            predictions.append((next_word, probability, count))
        
        return predictions
    
    def calculate_probability(self, word1, word2):
        """Calculate P(word2|word1)"""
        if word1 not in self.bigram_counts:
            return 0.0
        count_w1_w2 = self.bigram_counts[word1][word2]
        count_w1 = self.unigram_counts[word1]
        return count_w1_w2 / count_w1 if count_w1 > 0 else 0.0

# Train bigram model
bigram_model = BigramModel()
bigram_model.train(tokens)

print("Bigram Model Trained Successfully!")
print("=" * 70)
print("Testing bigram predictions:\n")

test_words = ['natural', 'language', 'machine', 'artificial', 'processing']
for word in test_words:
    predictions = bigram_model.predict_next(word, top_k=5)
    print(f"Word: '{word}'")
    if predictions:
        print("  Top 5 predictions:")
        for next_word, prob, count in predictions:
            print(f"    {next_word:20s} | Probability: {prob:.3f} | Count: {count}")
    else:
        print("  No predictions available")
    print()

## 5. Trigram Language Model

Building a more sophisticated trigram model.

In [ ]:
class TrigramModel:
    def __init__(self):
        self.trigram_counts = defaultdict(Counter)
        self.bigram_counts = defaultdict(Counter)
    
    def train(self, tokens):
        """Train the trigram model"""
        # Count bigrams (for context)
        for i in range(len(tokens) - 1):
            self.bigram_counts[tokens[i]][tokens[i+1]] += 1
        
        # Count trigrams
        for i in range(len(tokens) - 2):
            context = (tokens[i], tokens[i+1])
            self.trigram_counts[context][tokens[i+2]] += 1
    
    def predict_next(self, word1, word2, top_k=5):
        """Predict next word given two previous words"""
        context = (word1, word2)
        if context not in self.trigram_counts:
            return []
        
        next_words = self.trigram_counts[context]
        total = sum(next_words.values())
        
        predictions = []
        for next_word, count in next_words.most_common(top_k):
            probability = count / total
            predictions.append((next_word, probability, count))
        
        return predictions

# Train trigram model
trigram_model = TrigramModel()
trigram_model.train(tokens)

print("Trigram Model Trained Successfully!")
print("=" * 70)
print("Testing trigram predictions:\n")

test_contexts = [
    ('natural', 'language'),
    ('machine', 'learning'),
    ('artificial', 'intelligence'),
    ('language', 'processing'),
    ('deep', 'learning')
]

for word1, word2 in test_contexts:
    predictions = trigram_model.predict_next(word1, word2, top_k=5)
    print(f"Context: '{word1} {word2}'")
    if predictions:
        print("  Top predictions:")
        for next_word, prob, count in predictions:
            print(f"    {next_word:20s} | Probability: {prob:.3f} | Count: {count}")
    else:
        print("  No predictions available")
    print()

## 6. Complete Auto-complete System

Integrating N-gram models for intelligent auto-completion.

In [ ]:
class NgramAutoComplete:
    def __init__(self, bigram_model, trigram_model):
        self.bigram_model = bigram_model
        self.trigram_model = trigram_model
    
    def autocomplete(self, text, top_k=5):
        """Auto-complete the given text using N-gram models"""
        words = word_tokenize(text.lower())
        
        if len(words) == 0:
            return []
        
        # Use trigram model if we have at least 2 words
        if len(words) >= 2:
            trigram_preds = self.trigram_model.predict_next(words[-2], words[-1], top_k)
            if trigram_preds:
                return [(pred[0], pred[1], 'trigram') for pred in trigram_preds]
        
        # Fall back to bigram model
        if len(words) >= 1:
            bigram_preds = self.bigram_model.predict_next(words[-1], top_k)
            if bigram_preds:
                return [(pred[0], pred[1], 'bigram') for pred in bigram_preds]
        
        return []
    
    def generate_text(self, seed_text, num_words=10):
        """Generate text using the N-gram model"""
        words = word_tokenize(seed_text.lower())
        generated = list(words)
        
        for _ in range(num_words):
            # Try trigram first
            if len(generated) >= 2:
                preds = self.trigram_model.predict_next(generated[-2], generated[-1], top_k=1)
                if preds:
                    generated.append(preds[0][0])
                    continue
            
            # Fall back to bigram
            if len(generated) >= 1:
                preds = self.bigram_model.predict_next(generated[-1], top_k=1)
                if preds:
                    generated.append(preds[0][0])
                else:
                    break
            else:
                break
        
        return ' '.join(generated)

# Create auto-complete system
autocomplete = NgramAutoComplete(bigram_model, trigram_model)

print("N-gram Auto-complete System")
print("=" * 70)

# Test auto-completion
test_inputs = [
    "natural language",
    "machine learning",
    "artificial intelligence",
    "language processing",
    "the quick"
]

print("\nAuto-completion Tests:")
for text in test_inputs:
    predictions = autocomplete.autocomplete(text, top_k=5)
    print(f"\nInput: '{text}'")
    print("  Suggestions:")
    if predictions:
        for word, prob, model_type in predictions:
            print(f"    {word:20s} | Probability: {prob:.3f} | Model: {model_type}")
    else:
        print("    No suggestions available")

## 7. Text Generation

Generating complete sentences using the N-gram model.

In [ ]:
print("Text Generation using N-gram Models:")
print("=" * 70)

seed_texts = [
    "natural language",
    "machine learning",
    "artificial intelligence",
    "deep learning",
    "the quick"
]

for seed in seed_texts:
    generated = autocomplete.generate_text(seed, num_words=8)
    print(f"\nSeed: '{seed}'")
    print(f"Generated: {generated}")

## 8. Model Evaluation and Perplexity

Evaluating the N-gram model performance.

In [ ]:
def calculate_perplexity(model, test_tokens):
    """Calculate perplexity for bigram model"""
    log_prob = 0
    count = 0
    
    for i in range(len(test_tokens) - 1):
        prob = model.calculate_probability(test_tokens[i], test_tokens[i+1])
        if prob > 0:
            log_prob += np.log(prob)
            count += 1
    
    if count == 0:
        return float('inf')
    
    perplexity = np.exp(-log_prob / count)
    return perplexity

# Calculate perplexity on training data
perplexity = calculate_perplexity(bigram_model, tokens[:100])

print("Model Evaluation:")
print("=" * 70)
print(f"Bigram Model Perplexity: {perplexity:.2f}")
print(f"\nLower perplexity indicates better model performance.")
print(f"The model has learned {len(bigram_model.bigram_counts)} unique contexts.")

## 9. Interactive Auto-complete Demo

Demonstrating the auto-complete system with various inputs.

In [ ]:
def demonstrate_autocomplete(partial_text):
    """Demonstrate auto-complete with detailed output"""
    print(f"\nInput: '{partial_text}'")
    print("-" * 70)
    
    suggestions = autocomplete.autocomplete(partial_text, top_k=5)
    
    if not suggestions:
        print("No suggestions available for this input.")
        return
    
    print(f"{'Rank':<6}{'Suggestion':<25}{'Probability':<15}{'Model':<10}")
    print("-" * 70)
    
    for i, (word, prob, model_type) in enumerate(suggestions, 1):
        completed = f"{partial_text} {word}"
        print(f"{i:<6}{word:<25}{prob:<15.3f}{model_type:<10}")
    
    # Show most likely completion
    best_word = suggestions[0][0]
    print(f"\nMost likely completion: '{partial_text} {best_word}'")

print("Interactive Auto-complete Demonstration:")
print("=" * 70)

demo_inputs = [
    "natural language",
    "machine",
    "artificial",
    "language processing",
    "natural"
]

for input_text in demo_inputs:
    demonstrate_autocomplete(input_text)

## 10. Summary and Applications

In [ ]:
# Create summary statistics
summary_data = {
    'Model': ['Unigram', 'Bigram', 'Trigram'],
    'Total N-grams': [len(unigram_freq), len(bigram_freq), len(trigram_freq)],
    'Context Size': [0, 1, 2],
    'Prediction Accuracy': ['Low', 'Medium', 'High'],
    'Use Case': ['Basic completion', 'Word prediction', 'Phrase completion']
}

df_summary = pd.DataFrame(summary_data)

print("N-gram Model Summary:")
print("=" * 70)
print(df_summary.to_string(index=False))

print("\n" + "=" * 70)
print("\nApplications of N-gram Models:")
print("  • Auto-complete and predictive text")
print("  • Machine translation")
print("  • Speech recognition")
print("  • Spelling correction")
print("  • Text generation")
print("  • Authorship attribution")
print("  • Language modeling")

print("\n" + "=" * 70)
print("✓ N-gram Auto-complete System completed successfully!")